# V2 Band/Horizon Sensitivity Scan (Train-Domain, No Cell Selected)

**Designation (verbatim, in every artifact and summary):** train-domain
band/horizon sensitivity scan (train-inner domain evidence only; no cell
selected).

**Research question.** Does the frozen label cell (3.0 bps no-trade band, 9-bar
horizon) sit on a knife edge? The scan rebuilds train-segment labels at five
predeclared (band, horizon) cells — a CROSS, not a grid — runs the identical
train-inner machinery at each cell, and reads only whether the sign of the
same-row macro-F1 delta over each cell's own re-drawn stratified dummy is
stable across the adjacent cells on each axis.

**TUNING GUARD (verbatim, load-bearing).** This scan is NEVER a tuning pass.
No cell is preferred, cells are never ranked, and no alternative (band,
horizon) is recommended. The frozen protocol values (3.0 bps, 9 bars) remain
frozen regardless of outcome. Sign flips are reported honestly and strengthen
the paper's stated limitation.

**INVARIANT (bold, enforced in code):** the entire experiment runs on the
frozen TRAIN segment only (1998-01-02 through 2013-09-16, end-exclusive).
ZERO contact with the official validation split. ZERO contact with post-2017
rows. The stage code raises on any timestamp at or after 2013-09-16.

**Evidence status.** Every number this notebook produces is train-inner-domain
evidence about label-construction sensitivity. It is never fused with the
official validation, train-inner control, or guarded walk-forward domains, and
it is never model-selection evidence.

Preregistration (frozen before any cell runs):
`docs/protocols/v2_band_horizon_sensitivity_preregistration_20260701.md`

## Preregistration Summary

- Cells (frozen cross): band {2.0, 3.0, 4.0} bps at horizon 9; horizon
  {6, 9, 12} bars at band 3.0 — five cells including the frozen `h09_bps3p0`.
- Per cell: rebuild labels on the train segment with the frozen Stage 00 label
  operator at that (band, horizon) (label logic imported from
  `labels.make_direction_labels`, never forked), same feature set
  (`price_volume_time`), same window (w=20), the Stage 02 train-inner fold
  structure (3 chronological day-block folds), the frozen `tcn_tiny` profile
  `tcn_p01`, frozen seeds [101, 202], and the Stage 02 fold-row caps.
- Per-cell dummy (load-bearing): the same-row stratified dummy is RE-DRAWN per
  cell, per fold, per seed on that cell's own labels — each cell has its own
  label prior and eligible rows; no draw is shared across cells. The band
  changes eligibility mechanically, so eligible-row counts per cell are
  first-class outputs.
- Frozen-cell anchor gates: the rebuilt `h09_bps3p0` events must exactly match
  the frozen Stage 00 event index, the Stage 01 windowed counts, and the
  executed Stage 02 plan-ledger fold-row hashes — otherwise nothing is read.
- Predeclared reading (descriptive only): report all cells; "not a knife edge"
  = the delta sign is stable across all cells of both axes; any sign flip is
  reported per axis and strengthens the limitation; zero/undefined signs are
  reported as such; incomplete rows void the reading. Cells are NEVER ranked.
- Budget: 5 cells x 3 folds x 2 seeds = 30 tcn_tiny fits (cap 36).

FORBIDDEN wording in anything built on this run: best band, best horizon,
optimal band, optimal horizon, recommended band, recommended horizon, tuned
threshold, final model, clean test, profitable.

## Expected Artifacts

```
/content/lst_models_results/v2_band_horizon_sensitivity/<run_id>/
  run_manifest.json
  artifact_inventory.csv
  bhs_trial_ledger.csv
  bhs_cell_summary.csv
  bhs_cell_eligibility.csv
  bhs_fold_manifest.csv
  bhs_baseline_control_summary.csv
  bhs_reading_readout.json
  bhs_trials_h09_bps2p0.csv
  bhs_trials_h09_bps3p0.csv
  bhs_trials_h09_bps4p0.csv
  bhs_trials_h06_bps3p0.csv
  bhs_trials_h12_bps3p0.csv
```

Durable backup: `My Drive/lst_models/results/v2_band_horizon_sensitivity/<run_id>/`

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import subprocess
import sys
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
# Two-step exact-commit pin: push the full v2_band_horizon_sensitivity bundle
# (notebook + config + preregistration + src + tests) first, then fill
# PROJECT_REPO_COMMIT with that full-bundle commit.
PROJECT_REPO_COMMIT = "REPLACE_WITH_V2_BHS_FULL_BUNDLE_COMMIT"
PROJECT_ROOT = Path("/content/lst_models")

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_BHS = False
RUN_BHS_DRIVE_BACKUP = True
RUN_ARTIFACT_DISPLAY = False

STAGE_NAME = "v2_band_horizon_sensitivity"
ROUTE = "lst_models"
SCOPE = "validation_only"
HOLDOUT_TEST_CONTACT = False
TRAIN_DOMAIN_ONLY = True
TRAIN_END_EXCLUSIVE = "2013-09-16"
FROZEN_SEEDS = [101, 202]
SCAN_CELL_IDS = ["h09_bps2p0", "h09_bps3p0", "h09_bps4p0", "h06_bps3p0", "h12_bps3p0"]
FROZEN_CELL_ID = "h09_bps3p0"
FROZEN_HORIZON_K = 9
FROZEN_BAND_BPS = 3.0
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_RUN_ID = "20260610_082130_797479"
BHS_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_RUN_ID
OUTPUT_DIR = Path("/content/lst_models_results/v2_band_horizon_sensitivity")
BHS_OUTPUT_RUN_DIR = OUTPUT_DIR / BHS_RUN_ID
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_RUN_ID]
BHS_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_band_horizon_sensitivity"]
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/v2_band_horizon_sensitivity")

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "v2_band_horizon_sensitivity.yaml").exists()
        and (path / "docs" / "protocols" / "v2_band_horizon_sensitivity_preregistration_20260701.md").exists()
        and (path / "notebooks" / "v2_band_horizon_sensitivity_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "band_horizon_sensitivity.py").exists()
        and (path / "src" / "lst_models" / "robustness.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "REPLACE_WITH" in PROJECT_REPO_COMMIT:
            raise ValueError(
                "Fill PROJECT_REPO_COMMIT with the v2_band_horizon_sensitivity "
                "full-bundle commit before running (two-step exact-commit pin)."
            )
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_band_horizon_sensitivity.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_band_horizon_sensitivity_preregistration_20260701.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "v2_band_horizon_sensitivity_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "band_horizon_sensitivity.py"
DOMAIN_MODULE_PATH = PROJECT_ROOT / "src" / "lst_models" / "robustness.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
TCN_SEARCH_SPACE_PATH = PROJECT_ROOT / "configs" / "models" / "tcn" / "search_space.yaml"
REQUIRED_PROJECT_FILES = [
    STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH,
    DOMAIN_MODULE_PATH, RAW_DATA_MANIFEST_PATH, TCN_SEARCH_SPACE_PATH,
]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("v2_band_horizon_sensitivity bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BHS_RUN_ID:", BHS_RUN_ID)

## Config Load And Contract Check

In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the v2 band/horizon sensitivity config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required v2 band/horizon sensitivity file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    bhs_config = yaml.safe_load(handle)

stage_inputs = bhs_config["inputs"]
stage_outputs = bhs_config["outputs"]
stage_checkpointing = bhs_config["checkpointing"]
stage_inputs["stage00_run_id"] = STAGE00_RUN_ID
stage_inputs["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
stage_inputs["stage00_drive_path_parts"] = STAGE00_DRIVE_PATH_PARTS
stage_inputs["stage01_run_id"] = STAGE01_RUN_ID
stage_inputs["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
stage_inputs["stage01_drive_path_parts"] = STAGE01_DRIVE_PATH_PARTS
stage_inputs["stage02_run_id"] = STAGE02_RUN_ID
stage_inputs["stage02_runtime_run_dir"] = str(STAGE02_OUTPUT_DIR)
stage_inputs["stage02_drive_path_parts"] = STAGE02_DRIVE_PATH_PARTS
stage_inputs["raw_data_dir"] = str(RAW_DATA_DIR)
stage_outputs["output_dir"] = str(OUTPUT_DIR)
stage_outputs["run_id"] = BHS_RUN_ID
stage_checkpointing["checkpoint_dir"] = str(CHECKPOINT_ROOT)

assert bhs_config["stage_name"] == STAGE_NAME
assert bhs_config["route"] == ROUTE
assert bhs_config["scope"] == SCOPE
assert bhs_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert bhs_config["train_domain_only"] is TRAIN_DOMAIN_ONLY
assert bhs_config["sensitivity_scan_no_cell_selected"] is True
assert stage_inputs["stage00_run_id"] == STAGE00_RUN_ID
assert Path(stage_inputs["stage00_runtime_run_dir"]) == STAGE00_OUTPUT_DIR
assert stage_inputs["stage01_run_id"] == STAGE01_RUN_ID
assert Path(stage_inputs["stage01_runtime_run_dir"]) == STAGE01_OUTPUT_DIR
assert stage_inputs["stage02_run_id"] == STAGE02_RUN_ID
assert Path(stage_inputs["stage02_runtime_run_dir"]) == STAGE02_OUTPUT_DIR
assert Path(stage_inputs["raw_data_dir"]) == RAW_DATA_DIR
assert Path(stage_outputs["output_dir"]) == OUTPUT_DIR
assert stage_outputs["run_id"] == BHS_RUN_ID
assert Path(stage_checkpointing["checkpoint_dir"]) == CHECKPOINT_ROOT
assert stage_inputs["raw_data_manifest"] == "configs/lst_models_data.yaml"
assert bhs_config["candidate"]["candidate_id"] == "price_volume_time_w20"
assert bhs_config["model"]["family"] == "tcn"
assert bhs_config["model"]["probe_id"] == "tcn_tiny"
assert bhs_config["model"]["hpo_profile_id"] == "tcn_p01"
assert [cell["cell_id"] for cell in bhs_config["label_scan"]["cells"]] == SCAN_CELL_IDS
assert bhs_config["label_scan"]["frozen_cell"]["horizon_k"] == FROZEN_HORIZON_K
assert bhs_config["label_scan"]["frozen_cell"]["no_trade_band_bps"] == FROZEN_BAND_BPS
assert bhs_config["train_inner"]["n_folds"] == 3
assert bhs_config["train_inner"]["seeds"] == FROZEN_SEEDS
assert bhs_config["reading_rules"]["cells_are_never_ranked"] is True
assert bhs_config["reading_rules"]["no_alternative_cell_recommended"] is True
assert bhs_config["reading_rules"]["primary_baseline"] == "stratified_dummy_train_prior"
torch_defaults = bhs_config["probe_training_defaults"]["torch"]
assert torch_defaults["early_stopping"] == "inner_train_chronological_tail"
assert torch_defaults["epochs"] == 64
assert torch_defaults["gradient_clip_norm"] > 0
planned_fit_rows = (
    len(bhs_config["label_scan"]["cells"])
    * int(bhs_config["train_inner"]["n_folds"])
    * len(bhs_config["train_inner"]["seeds"])
)
assert planned_fit_rows <= bhs_config["budget"]["max_planned_fit_rows"]

from lst_models.stages.band_horizon_sensitivity import _validate_config
_validate_config(bhs_config)

print(json.dumps({
    "stage_name": bhs_config["stage_name"],
    "evidence_status": bhs_config["evidence_status"],
    "cells": SCAN_CELL_IDS,
    "frozen_cell_id": FROZEN_CELL_ID,
    "planned_fit_rows": planned_fit_rows,
    "candidate_id": bhs_config["candidate"]["candidate_id"],
    "profile": bhs_config["model"]["hpo_profile_id"],
    "seeds": bhs_config["train_inner"]["seeds"],
    "train_domain_only": bhs_config["train_domain_only"],
    "holdout_test_contact": bhs_config["holdout_test_contact"],
}, indent=2))

## Stage 00, Stage 01, Stage 02, And Raw Data Input Sync

Exact frozen run folders only (no latest-run scanning); raw `.txt` files are
downloaded by Drive file ID from `configs/lst_models_data.yaml`. The Stage 02
folder supplies only `02_hpo_plan_ledger.csv`, the fold-row parity source for
the frozen cell.

In [ ]:
from lst_models.artifacts import require_artifacts

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive", fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def get_drive_service_for_input_sync():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive sync only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    raw_source = raw_manifest["raw_source"]
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_source["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

required_stage00_artifacts = bhs_config["inputs"]["required_stage00_artifacts"]
required_stage01_artifacts = bhs_config["inputs"]["required_stage01_artifacts"]
required_stage02_artifacts = bhs_config["inputs"]["required_stage02_artifacts"]
if RUN_BHS:
    service = get_drive_service_for_input_sync()
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    missing_raw = []
    for ticker in raw_manifest["tickers"]:
        raw_path = RAW_DATA_DIR / raw_manifest["raw_source"]["files"][ticker]["name"]
        if not raw_path.exists():
            missing_raw.append(raw_path.name)
    if missing_raw and RUN_RAW_DATA_SYNC:
        print("Missing raw data files; syncing from Drive file IDs:", missing_raw)
        sync_raw_data_from_drive(service)
    stage00_missing = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if stage00_missing and RUN_STAGE00_DRIVE_SYNC:
        print("Missing Stage 00 artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, stage00_missing, "stage00")
    stage01_missing = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if stage01_missing and RUN_STAGE01_DRIVE_SYNC:
        print("Missing Stage 01 artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, stage01_missing, "stage01")
    stage02_missing = [name for name in required_stage02_artifacts if not (STAGE02_OUTPUT_DIR / name).exists()]
    if stage02_missing and RUN_STAGE02_DRIVE_SYNC:
        print("Missing Stage 02 plan-ledger artifacts; syncing exact frozen run from Drive.")
        sync_stage_artifacts_from_drive(service, STAGE02_OUTPUT_DIR, STAGE02_DRIVE_PATH_PARTS, stage02_missing, "stage02")
    require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    require_artifacts(STAGE02_OUTPUT_DIR, required_stage02_artifacts)
    print("Stage 00, Stage 01, Stage 02, and raw data input checks passed.")
else:
    print("RUN_BHS=False; input sync skipped.")

## Run The Band/Horizon Sensitivity Scan

Requires a GPU runtime (Runtime -> Change runtime type -> T4 GPU or better).
Expected wall clock on a T4: the raw rebuild plus five window-dataset builds
dominate; with 30 capped tcn_tiny fits the whole run is a single-session job
(order of one to three hours). One compact progress line prints per completed
cell via the checkpoint writer.

In [ ]:
if RUN_BHS:
    try:
        from lst_models.stages.band_horizon_sensitivity import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Missing entry point: src/lst_models/stages/band_horizon_sensitivity.py.") from exc
    result = run_stage(bhs_config)
    display(result)
    if Path(result.output_dir).name != BHS_RUN_ID:
        raise RuntimeError(f"run id mismatch: {Path(result.output_dir).name} != {BHS_RUN_ID}")
    if Path(result.output_dir) != BHS_OUTPUT_RUN_DIR:
        raise RuntimeError(f"output dir mismatch: {Path(result.output_dir)} != {BHS_OUTPUT_RUN_DIR}")
else:
    print("RUN_BHS=False; band/horizon sensitivity scan not executed.")

## Durable Drive Result Backup

Runs immediately after a successful `run_stage`. Validates the required
artifact list, uploads to the canonical Drive run folder, and writes
`drive_backup_manifest.json` last (self-reference recorded with null bytes).

In [ ]:
def get_drive_service_for_result_backup():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive backup only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_or_create_drive_folder(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    folder_mime = "application/vnd.google-apps.folder"
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and mimeType = '{folder_mime}' and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id, name)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive folders named {name!r} under parent {parent_id}; resolve manually")
    if matches:
        return matches[0]["id"]
    created = service.files().create(
        body={"name": name, "mimeType": folder_mime, "parents": [parent_id]}, fields="id"
    ).execute()
    return created["id"]

def upload_or_update_drive_file(service, folder_id, local_path):
    from googleapiclient.http import MediaFileUpload
    escaped_name = quote_drive_query_value(local_path.name)
    query = f"name = '{escaped_name}' and '{folder_id}' in parents and trashed = false"
    response = service.files().list(q=query, spaces="drive", fields="files(id, name)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) > 1:
        raise RuntimeError(f"duplicate Drive files named {local_path.name!r} under folder {folder_id}; resolve manually")
    media = MediaFileUpload(str(local_path), resumable=True)
    if matches:
        updated = service.files().update(fileId=matches[0]["id"], media_body=media, fields="id").execute()
        return updated["id"]
    created = service.files().create(
        body={"name": local_path.name, "parents": [folder_id]}, media_body=media, fields="id"
    ).execute()
    return created["id"]

if RUN_BHS_DRIVE_BACKUP and RUN_BHS:
    from lst_models.stages.band_horizon_sensitivity import REQUIRED_BHS_ARTIFACTS
    local_run_dir = BHS_OUTPUT_RUN_DIR
    per_cell_files = sorted(path.name for path in local_run_dir.glob("bhs_trials_*.csv"))
    required_backup_files = list(REQUIRED_BHS_ARTIFACTS) + per_cell_files
    missing_required_artifacts = [name for name in required_backup_files if not (local_run_dir / name).exists()]
    if missing_required_artifacts:
        raise FileNotFoundError(
            "Missing required v2_band_horizon_sensitivity artifacts before Drive backup: "
            f"{missing_required_artifacts} in {local_run_dir}"
        )
    service = get_drive_service_for_result_backup()
    parent_id = "root"
    for folder_name in BHS_DRIVE_RESULT_PATH_PARTS + [BHS_RUN_ID]:
        parent_id = find_or_create_drive_folder(service, parent_id, folder_name)
    drive_folder_id = parent_id
    uploaded_files = []
    for name in required_backup_files:
        local_path = local_run_dir / name
        drive_file_id = upload_or_update_drive_file(service, drive_folder_id, local_path)
        uploaded_files.append({
            "file_name": name,
            "relative_path": name,
            "drive_file_id": drive_file_id,
            "bytes": local_path.stat().st_size,
        })
        print("uploaded:", name)
    backup_manifest_path = local_run_dir / "drive_backup_manifest.json"
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": BHS_RUN_ID,
        "local_output_dir": str(local_run_dir),
        "drive_path_parts": BHS_DRIVE_RESULT_PATH_PARTS + [BHS_RUN_ID],
        "drive_folder_id": drive_folder_id,
        "uploaded_files": uploaded_files + [{
            "file_name": "drive_backup_manifest.json",
            "relative_path": "drive_backup_manifest.json",
            "drive_file_id": None,
            "bytes": None,
            "self_reference": True,
        }],
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "train_domain_only": True,
        "holdout_test_contact": False,
    }
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    manifest_file_id = upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)
    backup_manifest["uploaded_files"][-1]["drive_file_id"] = manifest_file_id
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    print("stage_run_id:", BHS_RUN_ID)
    print("drive_path:", "/".join(["My Drive"] + BHS_DRIVE_RESULT_PATH_PARTS + [BHS_RUN_ID]))
    print("drive_folder_id:", drive_folder_id)
else:
    print("RUN_BHS_DRIVE_BACKUP disabled or RUN_BHS=False; Drive backup skipped.")

## Artifact Display

In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd
    cell_summary = pd.read_csv(BHS_OUTPUT_RUN_DIR / "bhs_cell_summary.csv")
    display(cell_summary)
    eligibility = pd.read_csv(BHS_OUTPUT_RUN_DIR / "bhs_cell_eligibility.csv")
    display(eligibility[[
        "cell_id", "horizon_k", "no_trade_band_bps", "n_eligible_label_rows",
        "n_window_rows", "up_prior", "n_invalid_no_trade_band", "frozen_cell_event_parity",
    ]])
    with (BHS_OUTPUT_RUN_DIR / "bhs_reading_readout.json").open("r", encoding="utf-8") as handle:
        reading = json.load(handle)
    print(json.dumps({key: reading[key] for key in [
        "overall_outcome", "band_axis_sign_stable", "horizon_axis_sign_stable",
        "band_axis_signs", "horizon_axis_signs", "no_cell_preferred",
        "no_alternative_cell_recommended", "evidence_status",
    ]}, indent=2))
else:
    print("RUN_ARTIFACT_DISPLAY=False; artifact display skipped.")

## Interpretation Guard

- Everything above is **train-inner-domain evidence about label-construction
  sensitivity**. It is not market evidence, selects no model, and is never
  fused with the official validation, train-inner control, or guarded
  walk-forward domains.
- This scan is not a tuning pass. No cell is preferred, cells are never
  ranked, and no alternative (band, horizon) is recommended; the frozen
  protocol values (3.0 bps, 9 bars) remain frozen whatever the outcome.
- The only admissible reading is the preregistered sign-stability rule in
  `bhs_reading_readout.json`; sign flips strengthen the paper's stated
  limitation and authorize no re-scan.
- FORBIDDEN wording in any summary built on this run: best band, best
  horizon, optimal band, optimal horizon, recommended band, recommended
  horizon, tuned threshold, final model, clean test, profitable.
- Outcomes map to paper edits ONLY through the preregistration's section 10
  outcome map and the claims-ledger process; numbers may be quoted only from
  `bhs_cell_summary.csv` / `bhs_reading_readout.json` of the completed run.
- Any deviation (config edit, cell change, rerun) must be recorded in the
  preregistration's section 11 deviation log BEFORE results are interpreted.